In [1]:
import os
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnablePassthrough

from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader, TextLoader, DirectoryLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

load_dotenv()

c:\Users\mendh\Langchain-Langgraph\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


False

In [10]:
loader = DirectoryLoader(r'C:\Users\mendh\Langchain-Langgraph\0-Dataingestion-parsing\data\text_files', glob='**/*.txt',loader_cls=TextLoader, show_progress=True)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

100%|██████████| 4/4 [00:00<00:00, 1255.22it/s]


In [11]:
os.environ["Groq_API_KEY"] = os.getenv("Groq_API_KEY")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

C:\Users\mendh\AppData\Local\Temp\ipykernel_8180\4269086623.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
from langchain.chat_models import init_chat_model
chat_model = init_chat_model(model_provider="groq",model="llama-3.1-8b-instant", temperature=0.7)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
chat_model.invoke("Hello, how are you?")

AIMessage(content="I'm doing well, thank you for asking. I'm a large language model, so I don't have feelings or emotions like humans do, but I'm functioning properly and ready to assist you with any questions or tasks you may have. How can I help you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 41, 'total_tokens': 97, 'completion_time': 0.05592904, 'completion_tokens_details': None, 'prompt_time': 0.001964761, 'prompt_tokens_details': None, 'queue_time': 0.045768238, 'total_time': 0.057893801}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d424d-8a88-7422-b3b7-69f475a8ff36-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 56, 'total_tokens': 97})

In [15]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer the question based on the following retrieved documents: {context}"),
        ("human", "{input}")
    ]
)

In [17]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
combine_documents_chain = create_stuff_documents_chain(chat_model, prompt)
qa_chain = create_retrieval_chain(retriever, combine_documents_chain)
qa_chain.invoke({"input": "What is Machine Learning?"})

{'input': 'What is Machine Learning?',
 'context': [Document(id='b77644c4-838b-43f8-b1d5-8a53f28f6d82', metadata={'source': 'C:\\Users\\mendh\\Langchain-Langgraph\\0-Dataingestion-parsing\\data\\text_files\\MachineLearning.txt'}, page_content='Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit programming language instructions.[1] Within a subdiscipline of machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.'),
  Document(id='dff93ce7-b286-4596-8f2c-ecf01bf93a4d', metadata={'source': 'C:\\Users\\mendh\\Langchain-Langgraph\\0-Dataingestion-parsing\\data\\text_files\\MachineLearning.txt'}, page_content='Machine learning (ML), reorganised and recognised as its own field, star